# Step 4 prep: diagnostics for `case2s`

One-off inspections of the swap pipeline. Each diagnostic = one markdown cell (question → red flag → result) + one code cell that calls a function from `helpers/case2s_diagnostics.py`. Run after `03_case2_street_swap.ipynb` (§14 saves the frames read here).

## What each frame is
| Frame | One row is | Source |
|---|---|---|
| `pairs` | one labelled pair of the original set, all `neg_type`s | `load_all_pairs("overture")` |
| `written` | one pair of the new `case2s` set (originals + swaps) | `load_written()` |
| `street_pairs` | one `(street_a, street_b)` combination in the same split and ZIP, with the §6–§9 flags and `jw_similarity` | `Data/case2s_street_pairs.parquet` (03 §14) |
| `swaps` | one swap pair before the per-pair cap, with `jw` and `swap_type` | `Data/case2s_swaps.parquet` (03 §14) |

## Setup

In [2]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.pairs import load_all_pairs
from helpers.case2s_checks import load_written
from helpers.case2s_diagnostics import (
    row_gap, jw_band_sample, real_case2_jw, jw_quantiles, b_string_profile, concentration, only_prefix, only_prefix_sample,
)

SEED = 0
pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

## Load
**Output:** `pairs`, `written`, `street_pairs`, `swaps`

In [3]:
# Load: the frames the diagnostics read
pairs = load_all_pairs("overture")
written = load_written()
street_pairs = pd.read_parquet("Data/case2s_street_pairs.parquet")
swaps = pd.read_parquet("Data/case2s_swaps.parquet")
print(pairs.shape, written.shape, street_pairs.shape, swaps.shape)

(288277, 9) (288277, 9) (389286, 13) (18226, 15)


### D0 · Why does `pairs` have 288,277 rows when 02's result note says 288,294?
**Run after:** Load  
**How:** compare per split × `neg_type` against 02's result table; count `pairs["zip_b"].isna().sum()`  
**Red flag:** the 17-row gap isn't explained by the 17 NULL `zip_b` rows  
**Result (run by Claude, 2026-10-29):** all 17 missing rows are in train (`case0` −2, `case1` −4, `case3` −11; val and test match 02's table exactly), and `pairs` now has 0 NULL `zip_b`. That matches `logs/progress-history.md` (2026-10-11): the 17 NULL-`zip_b` rows, all in train, were dropped in 02 §8b after the result note was written, so the note's counts are pre-drop. Red flag not triggered. Not checkable row by row: the pre-drop files no longer exist. 02's note ("`zip_b` has 17 NULLs") is stale.

In [4]:
# D0: Why does `pairs` have 288,277 rows when 02's result note says 288,294?
display(row_gap(pairs))              # which cells lost rows
print(pairs["zip_b"].isna().sum())   # NULL zip_b still in pairs

neg_type,case0,case1,case2,case3
split,,,,
train,-2,-4,0,-11
val,0,0,0,0
test,0,0,0,0


0


### D1 · Do high-JW and low-JW street pairs look different?
**Run after:** §8, §9  
**How:** seeded samples from the top and bottom JW bands, per `swap_type`  
**Red flag:** high band = still the same street; low band = not really look-alike  
**Result (2026-10-29):** seeded sample of 5 pairs per 0.05-wide JW bin from 0.60 to 1.0. Below about 0.75 the pairs are plainly different streets; between 0.8 and 0.85 they look alike but are still different streets, so the 0.8 cutoff holds. The high band hits the red flag: same-street pairs survive the filters (`PLEASANT VALLY RD` / `PLEASANT VALLEY ROAD`, `225 GREENWICH AVENUE` / `GREENWICH AVENUE`, `NORTH COLONY ROAD` / `NORTH COLONY ED`), and raising the cutoff would not remove them. Accepted as a known limit (David's call, 2026-10-29): 5 rows per bin show the failure exists, not how often, and the sample was not split by `swap_type`.

In [5]:
# D1: Do high-JW and low-JW street pairs look different?
jw_band_sample(street_pairs, SEED)

,street_a,street_b,jw_similarity
108644,GROVE ST,MAIN ST,0.601190
350913,NORWICH RD,WINSOR AVE,0.622222
193467,HAYDEN STATION RD,LAMBERTON RD,0.629085
286872,WALLACE ST,GROVE ST,0.633333
63874,CHERRY ST,MELBA ST,0.648148
279571,HOWE ST,TOWER PKWY,0.657143
300386,BRIDGEPORT AVE STE 109,GROVE ST,0.662879
244316,BALDWIN ST,MILL ST,0.671429
277282,TALMADGE AVE,TOWNSEND AVE,0.674603
383333,NEW BRITAIN AVE,BERLIN TURNPIKE,0.677778


### D2 · Are the swaps about as hard as real case2?
**Run after:** §11  
**How:** JW distribution of `case2s` vs real `case2` (same clean-street JW)  
**Red flag:** `case2s` far above or below real case2  
**Result (2026-10-29):** the swaps are about as hard as real `case2`. Quartiles match (25%: 0.830 vs 0.829; 75%: 0.926 vs 0.926); median 0.841 vs 0.868 and 95% 0.939 vs 0.956 are slightly higher for `case2s`. The only gap is the lower tail: swaps start at 0.8 by construction, and 11.3% of real `case2` lies below it. Red flag not triggered. Real `case2` has only 221 rows, so this is a rough check. What the swaps add is volume, not difficulty.

In [6]:
# D2: Are the swaps about as hard as real case2?
real = real_case2_jw(pairs)
display(jw_quantiles(real, swaps))
print((real["jw"] > 0.8).mean())   # share of real case2 above the swap floor

,case2,case2s
0.05,0.641414,0.804978
0.25,0.829643,0.828571
0.50,0.841270,0.867857
0.75,0.925926,0.925926
0.95,0.939394,0.955556


0.8868778280542986


### D3 · Do the swapped B strings look like real Foursquare strings?
**Run after:** §11  
**How:** B length; share with a suite / unit / town: `case2s` vs `case0` B strings  
**Red flag:** a clear difference (a trace the model could learn)  
**Result (2026-10-29):** `case2s` B vs the `case0` B side: mean length 16.0 vs 14.8, median 15 vs 14; share with a suite / unit marker 1.05% vs 0.35%; share with state / ZIP text 0.72% vs 0.70%. State / ZIP matches. Length differs by about one character. The suite / unit share is 3× higher in `case2s` but tiny in absolute terms (about 1 row in 100), so it is at most a weak cue. David's read: the strings match; red flag not triggered. (Numbers added when D3 moved into the functions in `05`.)

In [7]:
# D3: Do the swapped B strings look like real Foursquare strings?
b_string_profile(swaps, pairs)

,case2s,case0 B
mean length,15.990343,14.827969
median length,15.000000,14.000000
share with suite / unit,0.010534,0.003503
share with state / ZIP,0.007188,0.007041


### D4 · Is the slice concentrated?
**Run after:** §11  
**How:** rows per X and per street pair; top 10 of each  
**Red flag:** a few streets or places supply most rows  
**Result (2026-10-29, on the final written `case2s` files, after the split / `id_a != id_b` / dedupe fixes):**

| split | swaps | distinct X | max rows per X | top-10 X share | street pairs | max rows per pair | top-10 pair share |
|---|---|---|---|---|---|---|---|
| train | 5,441 | 3,925 | 9 | 1.2% | 2,448 | 5 | 0.9% |
| val | 894 | 646 | 7 | 5.6% | 433 | 5 | 5.6% |
| test | 1,053 | 737 | 9 | 6.3% | 576 | 5 | 4.7% |

Red flag not triggered: no street pair or place supplies more than 6.3% of a split's swaps. The pair cap works (max 5 = `K`). There is no per-X cap in §13: one X can appear up to 9 times (David: fine).

In [8]:
# D4: Is the slice concentrated? (on the written case2s files: rows per X and per street pair)
concentration(written).round(3)

,swaps,distinct_x,max_per_x,top10_x_share,street_pairs,max_per_pair,top10_pair_share
split,,,,,,,
train,5441,3925,9,0.012,2448,5,0.009
val,894,646,7,0.056,433,5,0.056
test,1053,737,9,0.063,576,5,0.047


### D5 · Do the filters drop the right rows?
**Run after:** §3, §6, §7  
**How:** seeded sample of what each filter dropped  
**Red flag:** a dropped pair that should have stayed, or a kept one that should have gone  
**Result (2026-10-29), partial:** `prefix_match` checked: 1,651 pairs are removed only by `prefix_match`, and in the seeded sample of 40 every pair is the same street with extra text (`HALLS RD BOX 509` / `HALLS ROAD`, `CHASE AVE` / `CHASE AVE ZONE 1`, `HARTFORD TURNPIKE` / `HARTFORD TURNPIKE CT 06385 USA`). It drops the right rows. Not checked (David's call, moving on): `lib_match` (§6), `dir_disq` (§7), the §3 filter. The "kept one that should have gone" half is covered by D1 (same-street pairs in the high JW band).

In [9]:
# D5: Do the filters drop the right rows?
# Over-drop check for `prefix_match`: pairs that ONLY prefix_match removes, i.e. every other filter keeps them
print(len(only_prefix(street_pairs)), "pairs removed only by prefix_match")
print(only_prefix_sample(street_pairs, SEED))

1651 pairs removed only by prefix_match
                                       street_a                        street_b  jw_similarity     swap_type
50243       E AVENUE DERMATOLOGY FOR THE FAMILY                        E AVENUE       0.845714  name_differs
131481                         HALLS RD BOX 509                      HALLS ROAD       0.870278  name_differs
368424                    HAZARD AVE BUILDING 6                   HAZARD AVENUE       0.882234  name_differs
382546                       WILLARD AVE BLDG 1                     WILLARD AVE       0.922222  name_differs
385296                                CHASE AVE                CHASE AVE ZONE 1       0.912500  name_differs
210650               FARMINGTON AVE LOWER LEVEL                  FARMINGTON AVE       0.907692  name_differs
134207  BOSTON POST RD STE 300 GUILFORD COMMONS                  BOSTON POST RD       0.871795  name_differs
278844             SARGENT DR MOBIL GAS STATION                      SARGENT DR       0.